# 4D time-lapse: a CO2 plume

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_timelapse_4d_co2.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 6.5 Synthetic AVO Gathers
2. 10.4 Capstone: CO2 Plume Time-Lapse
3. 9.5 Export Your Model

In [ ]:
# Ask Ogbon starter: Monitor fluid change over time (4D)
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
SCENARIO = 'co2'
DVP_FRAC = -0.080   # reservoir Vp change from 40% saturation change
NRMS = 0.20        # survey repeatability noise
NX, NZ = 140, 100
F_HZ = 28.0
SEED = 7

In [ ]:
def ricker(f, dt, n):
    t = (np.arange(n) - n//2) * dt
    a = (np.pi * f * t) ** 2
    return (1 - 2*a) * np.exp(-a)

rng = np.random.default_rng(SEED)
w = ricker(F_HZ, 0.002, 61)

def section(ai):
    rc = np.zeros_like(ai)
    rc[1:, :] = (ai[1:, :] - ai[:-1, :]) / (ai[1:, :] + ai[:-1, :])
    return np.apply_along_axis(lambda c: np.convolve(c, w, 'same'), 0, rc)

ai = np.full((NZ, NX), 5000.0)
ai[30:, :] = 7200.0; ai[55:, :] = 6100.0; ai[80:, :] = 8600.0
res = (slice(55, 68), slice(35, 105))   # the reservoir zone

base = section(ai)
ai_mon = ai.copy()
ai_mon[res] *= (1.0 + DVP_FRAC)         # the fluid change
mon = section(ai_mon)

diff = mon - base
noise = rng.normal(0, 1, diff.shape) * NRMS * np.sqrt((base**2).mean())
observed = diff + noise

sig = np.sqrt((diff[res]**2).mean())
nz = np.sqrt((noise**2).mean())
print('4D signal / noise = %.2f -> %s' % (sig / nz,
      'DETECTABLE' if sig / nz >= 3 else 'MARGINAL' if sig / nz >= 1.5 else 'LOST IN NOISE'))

fig, ax = plt.subplots(1, 3, figsize=(11, 4))
for a2, d, ttl in zip(ax, [base, mon, observed], ['baseline', 'monitor', 'difference + noise']):
    m = np.abs(base).max()
    a2.imshow(d, cmap='seismic', vmin=-m, vmax=m, aspect='auto')
    a2.set_title(ttl)
plt.tight_layout(); plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)